# 05 — LLM Agent (Evidence-Traced Graph QA)

**Pipeline Step 5 of 5**

This notebook demonstrates the LLM-powered QA agent that queries the Micro-CKG with strict evidence traceability. Every answer cites exact `(Source)--[Edge_Type, Score=X.XX]-->(Target)` graph evidence.

### Hardened Guardrails
1. **No External Knowledge** — answers from graph context only
2. **Missing Data Fallback** — explicit "No evidence found" response
3. **Mandatory Citation** — `[Evidence: (Source) --(Edge)--> (Target)]`
4. **Objective Tone** — no speculation

### Prerequisites
- Ollama installed: `brew install ollama`
- Model pulled: `ollama pull llama3.1:8b`
- Server running: `ollama serve`

### Inputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG from Step 04 |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.biocypher_adapter import load_graph
from src.llm_agent import create_qa_agent, query_graph

CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 5.1 Load Micro-CKG

We load the persisted GraphML file produced by notebook 04. The graph is deserialized back into a NetworkX DiGraph with all node and edge attributes intact. The summary below confirms the graph structure matches expectations (number of gene nodes, cell-type nodes, region nodes, and total edges).

In [2]:
graph = load_graph(CACHE_DIR / "micro_ckg.graphml")
print(f"\nMicro-CKG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

  Graph loaded: 63 nodes, 601 edges

Micro-CKG: 63 nodes, 601 edges


## 5.2 Create QA Agent (Ollama — Local LLM)

The QA agent uses **Ollama** with `llama3.1:8b` (128K context) for fully local, free inference. The system prompt contains the serialized graph data and strict evidence-traceability rules.

To switch providers, change `provider` to `"google"` or `"openai"` and set the corresponding API key.

In [3]:
agent = create_qa_agent(graph, provider="ollama")
print("QA agent initialised (Ollama — llama3.1:8b).")

QA agent initialised (Ollama — llama3.1:8b).


## 5.3 Drug Target Discovery Queries

Three queries designed to extract actionable drug-development intelligence from the knowledge graph:

1. **Hub gene targets** — Which genes are the most connected hubs, and what brain regions are they specific to? Hub genes are high-value drug targets because they influence many downstream pathways.
2. **Region-specific pathways** — What differentiates cortical vs white-matter gene programs? Regional specificity determines whether a therapeutic can be targeted to a specific brain compartment.
3. **Druggable target summary** — Synthesize the top drug candidates by combining spatial specificity, network centrality, and cell-type associations into an actionable target list.

In [ ]:
# Query 1: Hub gene drug targets
answer1 = query_graph(
    agent,
    "Which genes have the highest number of connections (edges) in the graph "
    "and what brain regions are they expressed in? For each hub gene, explain "
    "why it could be a potential drug target for Alzheimer's disease."
)
print(answer1)

  Querying agent: Which genes have the highest number of connections (edges) in the graph and what...
To answer this question, we need to analyze the graph and identify the genes with the highest number of connections (edges). We will then examine the brain regions where these genes are expressed and explain why they could be potential drug targets for traumatic brain injury.

From the Micro-CKG, we can see that the genes with the highest number of connections are:

1. Tbr1 (34 connections)
2. Wnt4 (32 connections)
3. Trim54 (29 connections)
4. Ucn3 (26 connections)
5. Tbr1 (24 connections)

These genes are expressed in various brain regions, including:

1. Tbr1: Cortex, White Matter, Thalamus
2. Wnt4: White Matter, Thalamus, Hippocampus
3. Trim54: Cortex
4. Ucn3: White Matter
5. Tbr1: Cortex, White Matter, Thalamus

Now, let's examine why these genes could be potential drug targets for traumatic brain injury:

1. Tbr1: Tbr1 is a transcription factor that plays a crucial role in the de

In [5]:
# Query 2: Region-specific pathways
answer2 = query_graph(
    agent,
    "Compare the genes associated with Cortex cell types vs White_Matter cell types. "
    "Which genes are region-specific and which are shared? What biological pathways "
    "do these gene sets suggest for targeted therapy in each brain region?"
)
print(answer2)

  Querying agent: Compare the genes associated with Cortex cell types vs White_Matter cell types. ...
To answer this question, we need to analyze the Micro-CKG data and identify the genes associated with Cortex and White_Matter cell types.

From the Micro-CKG data, we can see that the following genes are associated with Cortex cell types:

* Tbr1 (associated with Cluster_0_Cortex, Cluster_1_Cortex, Cluster_7_Cortex)
* Trim54 (associated with Cluster_0_Cortex, Cluster_6_Cortex)
* Tafa2 (associated with Cluster_0_Cortex, Cluster_7_Cortex)

And the following genes are associated with White_Matter cell types:

* Wnt4 (associated with Cluster_10_White_Matter, Cluster_13_White_Matter, Cluster_14_White_Matter)
* Ucn3 (associated with Cluster_9_White_Matter)

Now, let's compare the genes associated with Cortex and White_Matter cell types.

**Region-specific genes:**

* Tbr1 is region-specific to Cortex cell types (associated with Cluster_0_Cortex, Cluster_1_Cortex, Cluster_7_Cortex)
* Trim54 i

In [ ]:
# Query 3: Druggable target summary
answer3 = query_graph(
    agent,
    "Summarize the top 5 most promising druggable targets from this spatial "
    "transcriptomics knowledge graph. For each gene, provide: (1) its stability score, "
    "(2) associated brain regions and cell types, (3) differential expression evidence "
    "(log2FC and p-value from edges), and (4) potential therapeutic relevance for "
    "Alzheimer's disease."
)
print(answer3)

  Querying agent: Summarize the top 5 most promising druggable targets from this spatial transcrip...
Based on the provided knowledge graph, I have identified the top 5 most promising druggable targets. Please note that the ranking is subjective and based on the available information.

**1. Wnt4**

* Stability score: 1.0
* Associated brain regions and cell types:
	+ White Matter (Cluster 10, 13, 14, 15, 16)
	+ Thalamus (Cluster 3)
	+ Hippocampus (Cluster 15)
* Differential expression evidence:
	+ Cluster 10: log2FC = -1.5752, p-value = 0.0
	+ Cluster 13: log2FC = 0.8233, p-value = 0.0
	+ Cluster 14: log2FC = 1.4639, p-value = 3e-06
	+ Cluster 15: log2FC = -1.7853, p-value = 0.012671
	+ Cluster 16: log2FC = -1.4417, p-value = 0.006623
* Potential therapeutic relevance for neurological diseases: Wnt4 is involved in various cellular processes, including cell proliferation and differentiation. Dysregulation of Wnt4 signaling has been implicated in several neurological disorders, such as Al